In [1]:
import numpy as np
import pandas as pd

# Plantilla

Es necesario ajustar las definiciones, las fuentes de los datos y posiblemente definiciones si la ENEMDU tiene una dimensión geográfica y temporal al mismo tiempo

In [2]:
# data = pd.read_stata(r"Z:\harmonized\ECU\ENEMDU\data_arm\ECU_1990m11_BID.dta") # para bases de stata
data = pd.read_stata(r"datos/ECU_1992m11_BID.dta") # para bases de stata

## Revisar los datos

- rn - región natural
- estrato - estrato
- cuidad - ciudad
- zona - zona
- sector - sector
- vivienda - vivienda
- hogar - hogar
- persona - persona
- numpers - número de personas
- edad - edad
- ingpat - Ingresos como patrono o cuenta propia
- ingasg - Ingresos como asalariado de gobierno
- ingepv - Ingresos asalariado empresa privada
- ingdom - Ingresos como empleada doméstica
- ingalq - Ingresos por alquileres, rentas o interese
- ingjub - Ingresos por jubilación o pensión
- ingotr - por otros ingresos
- fexp - factor de expansión
- ingrl - ingresos

El valor de 'ingasg' e 'ingepv' son el ingreso laboral monetario vamos a incluir 'ingdom' para darle un scope mayor, no hay datos sobre ingreso laboral no monetario, las otras variables son ingreso no laboral monetario y no monetario e ingrl es un ingreso total

Hay variables dicotomicas para cada mes (ene, feb, mar, abr, may, jun, jul, ago, sep, oct, nov, dic) que dicen si estuvo o no trabajando, se puede usar estas variables y el ingreso laboral asumiendo que cuando estaba trabajando tenía ese ingreso para intentar aproximar el salario mensual y de ahí el salario trimestral, esto solo funciona así ya que no tenemos una variable que explicite el mes, en encuestas que tengan el mes o trimestre explícito esto no sería igual.

In [3]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 38088 entries, 0 to 38087
Columns: 245 entries, region_BID_c to ppp_wdi2011
dtypes: category(77), float32(17), float64(109), int16(7), int32(1), int8(26), object(8)
memory usage: 40.9+ MB


Filtramos solo las columnas de interés para alivar el peso en la memoria

In [4]:
data.columns

Index(['region_BID_c', 'region_c', 'pais_c', 'anio_c', 'mes_c', 'zona_c',
       'factor_ch', 'idh_ch', 'idp_ci', 'factor_ci',
       ...
       'peamsiu', 'fexp1', 'fexp', 'contrato_ci', 'segsoc_ci', 'formal',
       'formal_1', 'ylm_ci1', 'ynlm_ci1', 'ppp_wdi2011'],
      dtype='object', length=245)

In [5]:
data = data[['rn', 'estrato', 'ciudad', 'zona', 'sector', 'vivienda', 'hogar', 'persona', 'numpers', 'edad', 'ingpat', 'ingasg','ingepv',
             'ingdom', 'ingalq', 'ingjub', 'ingotr', 'fexp', 'ingrl', 'ene', 'feb', 'mar', 'abr', 'may', 'jun', 'jul', 'ago', 'sep', 'oct', 'nov', 'dic']]

Creamos una variable de ingreso laboral que es igual al ingreso por asalariado en el sector público, asalariado en el sector privado o ingresos por empleo doméstico

In [6]:
data['ingasg'] = data['ingasg'].replace(9999999, np.nan)
data['ingepv'] = data['ingepv'].replace(9999999, np.nan)
data['ingdom'] = data['ingdom'].replace(9999999, np.nan)

In [7]:
data['ingr'] = data[['ingasg', 'ingepv', 'ingdom']].sum(axis=1, min_count=1)

Ingreso mensual asumiendo que las personas reciben el mismo valor reportado en 'ingr' siempre que reportan estar ocupados en un mes, ahora las variables categoricas por mes tienen diferentes leyendas
- trabajando
- buscando trabajo
- sin buscar trabajo

In [8]:
data['ingr_ene'] = data.apply(lambda x: x['ingr'] if x['ene'] == 'trabajando' else None, axis=1)
data['ingr_feb'] = data.apply(lambda x: x['ingr'] if x['feb'] == 'trabajando' else None, axis=1)
data['ingr_mar'] = data.apply(lambda x: x['ingr'] if x['mar'] == 'trabajando' else None, axis=1)
data['ingr_abr'] = data.apply(lambda x: x['ingr'] if x['abr'] == 'trabajando' else None, axis=1)
data['ingr_may'] = data.apply(lambda x: x['ingr'] if x['may'] == 'trabajando' else None, axis=1)
data['ingr_jun'] = data.apply(lambda x: x['ingr'] if x['jun'] == 'trabajando' else None, axis=1)
data['ingr_jul'] = data.apply(lambda x: x['ingr'] if x['jul'] == 'trabajando' else None, axis=1)
data['ingr_ago'] = data.apply(lambda x: x['ingr'] if x['ago'] == 'trabajando' else None, axis=1)
data['ingr_sep'] = data.apply(lambda x: x['ingr'] if x['sep'] == 'trabajando' else None, axis=1)
data['ingr_oct'] = data.apply(lambda x: x['ingr'] if x['oct'] == 'trabajando' else None, axis=1)
data['ingr_nov'] = data.apply(lambda x: x['ingr'] if x['nov'] == 'trabajando' else None, axis=1)
data['ingr_dic'] = data.apply(lambda x: x['ingr'] if x['dic'] == 'trabajando' else None, axis=1)

## Deflactamos y transformamos el ingreso

Esto deja todo en dólares constantes de 2014, utilizamos el IPC de Estados Unidos para ajustar por inflación ya que no podemos usar la inflación en sucres si queremos dejar el valor final en dólares de 2014

In [9]:
# Carga base de datos con ipc
data_externa = pd.read_excel("data_externa.xlsx", sheet_name='datos')

# filtra año de interés
datos_actual = data_externa[data_externa['Año'] == 1992]
datos_base = data_externa[data_externa['Año'] == 2014]

Diccionarios de ipc y tipo de cambio

In [10]:
ipc_dict = dict(zip(datos_actual['trimestre'], datos_actual['IPC Estados Unidos']))

ipc_base_dict = dict(zip(datos_base['trimestre'], datos_base['IPC Estados Unidos']))

tipo_cambio_dict = dict(zip(datos_actual['trimestre'], datos_actual['tipo de cambio']))

### Asignamos el ipc y tipo de cambio correspondiente según trimestre

$\begin{equation}
    ingr_{USD-base-2014}^{i} = \frac{ingr_{sucres}^{i}}{tipo-de-cambio^{i}}\left( \frac{ipcUSA^{i}_{2014}}{ipcUSA^{i}_{año}}\right)
\end{equation}$

Con $i$ el trimestre de interés y $año$ el año de interés

In [11]:
# Función que asigna valores correspondientes
def asigna_ipc(trimestre):
    return ipc_dict.get(trimestre, {})

def asigna_ipc_base(trimestre):
    return ipc_base_dict.get(trimestre, {})

In [12]:
data['ipc_t1'] = ipc_dict.get(1)
data['ipc_base_t1'] = ipc_base_dict.get(1)
data['tipo_cambio_t1'] = tipo_cambio_dict.get(1)

data['ipc_t2'] = ipc_dict.get(2)
data['ipc_base_t2'] = ipc_base_dict.get(2)
data['tipo_cambio_t2'] = tipo_cambio_dict.get(2)

data['ipc_t3'] = ipc_dict.get(3)
data['ipc_base_t3'] = ipc_base_dict.get(3)
data['tipo_cambio_t3'] = tipo_cambio_dict.get(3)

data['ipc_t4'] = ipc_dict.get(4)
data['ipc_base_t4'] = ipc_base_dict.get(4)
data['tipo_cambio_t4'] = tipo_cambio_dict.get(4)

In [13]:
# Calculamos el deflactor
data['def_t1'] = (data['ipc_base_t1'] / data['ipc_t1'])
data['def_t2'] = (data['ipc_base_t2'] / data['ipc_t2'])
data['def_t3'] = (data['ipc_base_t3'] / data['ipc_t3'])
data['def_t4'] = (data['ipc_base_t4'] / data['ipc_t4'])

In [14]:
# Ingreso real por mes
data['ingr_ene_r'] = (data['ingr_ene'] / data['tipo_cambio_t1']) * data['def_t1']
data['ingr_feb_r'] = (data['ingr_feb'] / data['tipo_cambio_t1']) * data['def_t1']
data['ingr_mar_r'] = (data['ingr_mar'] / data['tipo_cambio_t1']) * data['def_t1']
data['ingr_abr_r'] = (data['ingr_abr'] / data['tipo_cambio_t2']) * data['def_t2']
data['ingr_may_r'] = (data['ingr_may'] / data['tipo_cambio_t2']) * data['def_t2']
data['ingr_jun_r'] = (data['ingr_jun'] / data['tipo_cambio_t2']) * data['def_t2']
data['ingr_jul_r'] = (data['ingr_jul'] / data['tipo_cambio_t3']) * data['def_t3']
data['ingr_ago_r'] = (data['ingr_ago'] / data['tipo_cambio_t3']) * data['def_t3']
data['ingr_sep_r'] = (data['ingr_sep'] / data['tipo_cambio_t3']) * data['def_t3']
data['ingr_oct_r'] = (data['ingr_oct'] / data['tipo_cambio_t4']) * data['def_t4']
data['ingr_nov_r'] = (data['ingr_nov'] / data['tipo_cambio_t4']) * data['def_t4']
data['ingr_dic_r'] = (data['ingr_dic'] / data['tipo_cambio_t4']) * data['def_t4']

Ingreso mensual promedio en el trimeste

In [16]:
data['ingr_t1_r'] = (data['ingr_ene_r'] + data['ingr_feb_r'] + data['ingr_mar_r'])/3
data['ingr_t2_r'] = (data['ingr_abr_r'] + data['ingr_may_r'] + data['ingr_jun_r'])/3
data['ingr_t3_r'] = (data['ingr_jul_r'] + data['ingr_ago_r'] + data['ingr_sep_r'])/3
data['ingr_t4_r'] = (data['ingr_oct_r'] + data['ingr_nov_r'] + data['ingr_dic_r'])/3

## Regiones

In [17]:
# Corregimos los códigos para usarlos cómo texto
data['ciudad'] = data['ciudad'].apply(str)
data['ciudad'] = data['ciudad'].apply(lambda x: '0' + x if len(x) == 5 else x)

data['ciudad_2'] = data['ciudad'].apply(lambda x: x[:2])

In [18]:
regiones_dict = {
    'Guayas': '09',
    'Manabí': '13',
    'El Oro': '07',
    'Los Ríos': '12',
    'Pichincha': '17',
    'Azuay': '01',
    'Galápagos': '20',
    'Sierra': ['04', '10', '05', '18', '02', '06', '03', '11'],
    'Costa, Santo Domingo': ['08', '24', '23'],
    'Amazonía': ['14', '15', '16', '19', '21', '22', '90']
}

In [19]:
codigo_region = {}
for region, codes in regiones_dict.items():
    
    if isinstance(codes, list):
        for code in codes:
            codigo_region[code] = region
    
    else:
        codigo_region[codes] = region

# Mapeo de regiones
data['region'] = data['ciudad_2'].map(codigo_region)

In [20]:
data['region'].value_counts()

region
Guayas                  10130
Pichincha                6771
Sierra                   5493
El Oro                   3345
Azuay                    3148
Amazonía                 3012
Manabí                   2973
Los Ríos                 2112
Costa, Santo Domingo     1104
Name: count, dtype: int64

## Calculo ingreso de los hogares

In [21]:
columnas_idef = ['rn', 'estrato', 'ciudad', 'zona', 'sector', 'vivienda', 'hogar']

data['idef_hogar'] = data[columnas_idef].astype(str).agg(''.join, axis=1)
len(data['idef_hogar'].unique())

8129

In [22]:
data[['rn', 'estrato', 'ciudad', 'zona', 'sector', 'vivienda', 'hogar', 'idef_hogar', 'persona', 'numpers']]

,rn,estrato,ciudad,zona,sector,vivienda,hogar,idef_hogar,persona,numpers
0,1,3,010150,001,005,01,1,13010150001005011,1,3
1,1,3,010150,001,005,01,1,13010150001005011,2,3
2,1,3,010150,001,005,01,1,13010150001005011,3,3
3,1,3,010150,001,005,02,1,13010150001005021,1,4
4,1,3,010150,001,005,02,1,13010150001005021,2,4
...,...,...,...,...,...,...,...,...,...,...
38083,3,0,210450,001,011,10,1,30210450001011101,4,4
38084,3,0,210450,001,011,11,1,30210450001011111,1,4
38085,3,0,210450,001,011,11,1,30210450001011111,2,4
38086,3,0,210450,001,011,11,1,30210450001011111,3,4


Si todos los miembros del hogar tienen NA como ingreso, mantener NA, si al menos uno tiene un ingreso sumamos para el ingreso del hogar, así evitamos subestimar el ingreso del hogar si tenemos valores perdidos

In [23]:
# Definimos una función que sume pero devuelva NA si todos son NA
def sum_with_na(series):
    if series.isna().all():
        return pd.NA
    else:
        return series.sum(skipna=True)

In [24]:
data['ingr_t1_h'] = data.groupby('idef_hogar')['ingr_t1_r'].transform(sum_with_na)
data['ingr_t2_h'] = data.groupby('idef_hogar')['ingr_t2_r'].transform(sum_with_na)
data['ingr_t3_h'] = data.groupby('idef_hogar')['ingr_t3_r'].transform(sum_with_na)
data['ingr_t4_h'] = data.groupby('idef_hogar')['ingr_t4_r'].transform(sum_with_na)

In [25]:
data[['ingr_t1_h', 'ingr_t2_h', 'ingr_t3_h', 'ingr_t4_h']].mean()

ingr_t1_h    321.398421
ingr_t2_h    295.735146
ingr_t3_h    253.641891
ingr_t4_h    223.883501
dtype: object

In [26]:
print("Ingreso medio de un hogar t4: ", data['ingr_t4_h'].mean())
print("Mediana del ingreso de un hogar t4: ", data['ingr_t4_h'].median())

Ingreso medio de un hogar t4:  223.8835011676826
Mediana del ingreso de un hogar t4:  166.19768675977437


## Sacamos edades negativas y mayores a 100 años

In [27]:
len(data)

38088

En este caso en la variable edad tenemos números y el texto 'menos de un año' así que primero transformamos todas las filas que digan 'menos de un año' a 0

In [28]:
data['edad'] = data['edad'].apply(lambda x: x if type(x) == int else 0)

In [29]:
data = data.loc[(data['edad'] >= 0) & (data['edad'] < 100)]
len(data)

38088

## Ingreso individual descontando cargas familiares

Utilizando la metodología del autor dividimos el ingreso del hogar para la escala $(A_{i}+kC_{i})^{s}$ donde $A_{i}$ es al número de adultos, $C_{i}$ es el número de niños en el hogar $i$. $k$ es el costo en recursos de cada niño y $s$ busca reflejar las restricciones

In [30]:
k = 0.4
s = 0.9

In [31]:
# Si es necesario calcular el número de niños
data['es_nino'] = data['edad'] < 10

data['ninos'] = data.groupby('idef_hogar')['es_nino'].transform('sum')

# Si es necesario calcular el número de adultos
data['es_adulto'] = data['edad'] > 10

data['adultos'] = data.groupby('idef_hogar')['es_adulto'].transform('sum')

In [32]:
data['escala'] = (data['adultos'] + k * data['ninos']) ** s

In [33]:
data['ingr_t_t1'] = data['ingr_t1_h'] / data['escala']
data['ingr_t_t2'] = data['ingr_t2_h'] / data['escala']
data['ingr_t_t3'] = data['ingr_t3_h'] / data['escala']
data['ingr_t_t4'] = data['ingr_t4_h'] / data['escala']

In [34]:
data[['ingr_t_t1', 'ingr_t_t2', 'ingr_t_t3', 'ingr_t_t4']]

,ingr_t_t1,ingr_t_t2,ingr_t_t3,ingr_t_t4
0,<NA>,<NA>,<NA>,<NA>
1,<NA>,<NA>,<NA>,<NA>
2,<NA>,<NA>,<NA>,<NA>
3,69.495246,63.675645,54.413582,48.479394
4,69.495246,63.675645,54.413582,48.479394
...,...,...,...,...
38083,64.5313,59.127385,50.526897,45.01658
38084,<NA>,<NA>,<NA>,<NA>
38085,<NA>,<NA>,<NA>,<NA>
38086,<NA>,<NA>,<NA>,<NA>


In [35]:
print("Ingreso individual descontando cargas familiares t4: ", data['ingr_t_t4'].mean())
print("Mediana del ingreso individual descontando cargas familiares t4: ", data['ingr_t_t4'].median())

Ingreso individual descontando cargas familiares t4:  58.14573229699425
Mediana del ingreso individual descontando cargas familiares t4:  41.74153243111722


## Umbrales de pobreza

Incluimos los índices de pobreza si es posible a nivel regional para luego poder utilizar de mejor forma el factor de expansión

$\begin{equation}
    umbral_{USD-base-2014}^{i} = umbral_{año}\left( \frac{ipc^{i}_{2014}}{ipc^{i}_{año}}\right)
\end{equation}$

Con $i$ el trimestre de interés y $año$ el año de interés

Diccionario de umbral

In [36]:
umbral_dict = dict(zip(datos_actual['trimestre'], datos_actual['umbral de pobreza']))
salario_dict = dict(zip(datos_actual['trimestre'], datos_actual['salario básico unificado']))
ano = 1992

## Cálculo del índice de pobreza de Foster, Greer y Thorbecke

Para calcular un ínidce de pobreza se utiliza a Foster, Greer y Thorbecke (1984), ya que satisface algunas caracterísitcas de distribución que son positivas e igual a las enunciadas por Sen, el autor usa el mismo índice.

$\begin{equation}FGT_{\alpha} = \frac{1}{N}\sum_{i=1}^{H}\left(\frac{z-y_{i}}{z}\right)^{\alpha}\end{equation}$

Donde $z$ es el umbral de pobreza, $N$ es el número de personas en la economía, $H$ es el número de pobres (personas debajo de la línea de pobreza) $y_{i}$ es el ingreso de cada individuo. Mientras mayor es el valor de $\alpha$ mayor es el peso de los individuos más pobres, mayor $FGT$ mayor pobreza en la economía.

En este caso los umbrales están anivel nacional, aún así buscamos calcular la pobreza por región y sacar un promedio ponderado por región para la pobreza nacional, con el objetivo de hacerlo más específico

## Calculo del índice de desigualdad de Atkinson

Vamos a calcular el índice de desigualdad de atkinson con un parámetro $\epsilon$ de aversión a la desigualdad y un $\mu$ que es igual a la media de los ingresos individuales, con la siguiente fórmula.

$\begin{equation}A = 1-\frac{1}{\mu}\left(\frac{1}{N}\sum_{i=1}^{N}y^{1-\epsilon}\right)^{1/(1-\epsilon)}\end{equation}$

Donde $y_{i}$ es el ingreso individual y $\mu$ es el ingreso medio

In [37]:
resultados_list = []

# Para cada trimeste
for t in [1, 2, 3, 4]:
    salario = salario_dict.get(t)
    col_ingr = f'ingr_t_t{t}'
    umbral = umbral_dict.get(t)
    
    # Agrupa por región
    grouped = data.groupby('region')
    
    for region_name, group in grouped:
        # 1. Filtra datos
        valid = group.dropna(subset=[col_ingr])
        
        if len(valid) == 0:
            continue
            
        # Extrae los vectores 
        ingresos = valid[col_ingr].values
        pesos = valid['fexp'].values
        
        # 2. Calcula indices
        gaps = (umbral - ingresos) / umbral
        gaps = np.clip(gaps, a_min=0, a_max=None)
        
        # 3. Calcula FGT
        total_poblacion = pesos.sum()
        
        # FGT0
        fgt0 = (pesos * (gaps > 0).astype(int)).sum() / total_poblacion
        
        # FGT1
        fgt1 = (pesos * (gaps ** 1)).sum() / total_poblacion
        
        # FGT2
        fgt2 = (pesos * (gaps ** 2)).sum() / total_poblacion
        
        # 4. Calcula Ingreso promedio
        ingreso_promedio = np.average(ingresos, weights=pesos)

        # 5. Desigualdad de Atkinson
        atkinson_resultados = {}
        
        if ingreso_promedio > 0:
            for epsilon in [0.25, 0.5, 0.75]:
                # La suma ponderada de la utilidad
                utility_sum = np.sum((ingresos ** (1 - epsilon)) * pesos)
                
                # promedio de esa utilidad
                utility_mean = utility_sum / total_poblacion
                
                # ingreso equivalente
                y_ede = utility_mean ** (1 / (1 - epsilon))
                
                # índice final
                atkinson_index = 1 - (y_ede / ingreso_promedio)
                atkinson_resultados[f'a{int(epsilon*100)}'] = atkinson_index
        else:
            # Si nadie gana nada, definimos desigualdad como NaN
            atkinson_resultados = {'a25': np.nan, 'a50': np.nan, 'a75': np.nan}

        # 6. Calcula mediana del ingreso
        # Ordena
        sort_idx = np.argsort(ingresos)
        ingreso_ordenado = ingresos[sort_idx]
        pesos_ordenado = pesos[sort_idx]
        cumsum_pesos = np.cumsum(pesos_ordenado)
        cutoff = total_poblacion / 2.0
        mediana = ingreso_ordenado[np.searchsorted(cumsum_pesos, cutoff)]
        
        # 7. Guarda resultados
        resultados_list.append({
            'ano': ano,
            'trimestre': t,
            'region': region_name,
            'fgt0': fgt0,
            'fgt1': fgt1,
            'fgt2': fgt2,
            'a25': atkinson_resultados['a25'],
            'a50': atkinson_resultados['a50'],
            'a75': atkinson_resultados['a75'],
            'ingreso_promedio': ingreso_promedio,
            'ingreso_mediana': mediana,
            'salario_minimo': salario,
            'kaitz_indice': salario / mediana            
        })

# lista a DataFrame
df_final_regional = pd.DataFrame(resultados_list)
df_final_regional

,ano,trimestre,region,fgt0,fgt1,fgt2,a25,a50,a75,ingreso_promedio,ingreso_mediana,salario_minimo,kaitz_indice
0,1992,1,Amazonía,0.463453,0.196255,0.106877,0.056364,0.110532,0.162820,80.287936,67.304788,30.0,0.445734
1,1992,1,Azuay,0.475481,0.191351,0.107731,0.094687,0.177866,0.252092,103.031715,68.715533,30.0,0.436583
2,1992,1,"Costa, Santo Domingo",0.654008,0.328954,0.205382,0.080592,0.156679,0.228209,63.644136,44.672604,30.0,0.671553
3,1992,1,El Oro,0.517721,0.230655,0.139745,0.062626,0.125541,0.188963,75.306725,63.836034,30.0,0.469954
4,1992,1,Guayas,0.559738,0.240807,0.135082,0.086837,0.161418,0.227623,81.699136,58.074385,30.0,0.516579
5,1992,1,Los Ríos,0.600109,0.254748,0.143500,0.073663,0.141545,0.205033,72.468472,55.007852,30.0,0.545377
6,1992,1,Manabí,0.666705,0.316039,0.194262,0.083554,0.159857,0.230273,65.192178,48.895868,30.0,0.613549
7,1992,1,Pichincha,0.453637,0.173648,0.089792,0.092223,0.175268,0.249546,112.024731,72.018703,30.0,0.416558
8,1992,1,Sierra,0.576696,0.266483,0.158773,0.069735,0.138546,0.206203,73.734342,55.276709,30.0,0.542724
9,1992,2,Amazonía,0.544719,0.221972,0.122846,0.056727,0.111198,0.163715,73.906207,61.668618,40.0,0.648628


### Inserta los cálculos en la base final

In [38]:
indices = pd.read_csv("indices_region.csv", encoding='latin-1')

In [39]:
import os

# 2. cheque el archivo
if not os.path.isfile('indices_region.csv'):
    # Headers si es la primera vez
    df_final_regional.to_csv('indices_region.csv', index=False, encoding='latin-1')
else:
    # SI ya existe, append
    df_final_regional.to_csv('indices_region.csv', mode='a', index=False, header=False, encoding='latin-1')